In [1]:
# ============================================================
# NOTEBOOK 9 — POWER BI EXPORTS
# Generate the 8 CSV files that feed the Power BI dashboard.
# Each file is independent — failure in one does not break others.
# ============================================================

In [2]:
import pandas as pd
import os

In [3]:
# ------------------------------------------------------------
# PATH SETUP & LOAD CHECKPOINTS
# ------------------------------------------------------------
path = r'/Users/elia/Desktop/[02] Data Projects/[2] Working Folder/[3] InstaCart Store'

In [4]:
# Customer-level table (used for files 1–6)
df_customers_agg = pd.read_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'customers_agg_step7.pkl')
)
print(f"Customer aggregation loaded: {df_customers_agg.shape}")

Customer aggregation loaded: (206209, 27)


In [5]:
# Transaction-level segmented file (used for files 7 and 8)
df_master_segmented = pd.read_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'instacart_master_segmented.pkl')
)
print(f"Transaction-level segmented loaded: {df_master_segmented.shape}")
print()

Transaction-level segmented loaded: (32398592, 40)



In [6]:
# Output folder
output_dir = os.path.join(path, '[2] Data', '[3] Power BI Exports')
os.makedirs(output_dir, exist_ok=True)

In [7]:
# Helper to save and report
def save_csv(df, filename):
    fpath = os.path.join(output_dir, filename)
    df.to_csv(fpath, index=False)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"   ✅ {filename:<55} {df.shape[0]:>9,} rows × {df.shape[1]:>2} cols  ({size_mb:.2f} MB)")

In [8]:
# ============================================================
# FILE 1 — CUSTOMER SMART SEGMENTATION (the master export)
# ============================================================
# One row per customer with every segmentation field.
# Column order matches the existing Power BI export contract.

In [9]:
print("Building file 1...")

file_01 = df_customers_agg.rename(columns={
    'total_line_items': 'transaction_rows'
})[[
    'user_id',
    'gender', 'state', 'region', 'age', 'age_group',
    'income', 'income_group',
    'n_dependants', 'fam_status', 'household_group',
    'demographic_profile',
    'transaction_rows', 'unique_orders', 'unique_products',
    'avg_product_price', 'total_reordered_items', 'reorder_rate',
    'orders_pct_rank', 'products_pct_rank',
    'price_pct_rank', 'reorder_pct_rank',
    'behaviour_score', 'priority_level',
    'behaviour_segment', 'strategic_target_segment'
]].copy()

assert file_01.shape[0] == 206209
assert file_01.isnull().sum().sum() == 0

save_csv(file_01, '01_customer_smart_segmentation_FULL.csv')

Building file 1...
   ✅ 01_customer_smart_segmentation_FULL.csv                   206,209 rows × 26 cols  (71.02 MB)


In [10]:
# ============================================================
# FILE 2 — STRATEGIC SEGMENT SUMMARY
# ============================================================
# One row per strategic_target_segment with KPIs.

In [11]:
print()
print("Building file 2...")

file_02 = df_customers_agg.groupby('strategic_target_segment', observed=True).agg(
    customers              = ('user_id',               'nunique'),
    avg_orders_per_customer = ('unique_orders',         'mean'),
    median_orders_per_customer = ('unique_orders',     'median'),
    avg_unique_products    = ('unique_products',       'mean'),
    median_unique_products = ('unique_products',       'median'),
    avg_product_price      = ('avg_product_price',     'mean'),
    avg_reordered_items    = ('total_reordered_items', 'mean'),
    avg_reorder_rate       = ('reorder_rate',          'mean'),
    avg_income             = ('income',                'mean'),
    avg_age                = ('age',                   'mean'),
    avg_behaviour_score    = ('behaviour_score',       'mean'),
).reset_index()

file_02['customer_share'] = file_02['customers'] / file_02['customers'].sum()

assert file_02['customers'].sum() == 206209

save_csv(file_02, '02_strategic_segment_summary_FULL.csv')


Building file 2...
   ✅ 02_strategic_segment_summary_FULL.csv                          11 rows × 13 cols  (0.00 MB)


In [12]:
# ============================================================
# FILE 3 — PRIORITY LEVEL SUMMARY
# ============================================================

In [14]:
print()
print("Building file 3...")

priority_sort_map = {
    'High Priority':   1,
    'Growth Priority': 2,
    'Maintain':        3,
    'Low Priority':    4
}

file_03 = df_customers_agg.groupby('priority_level', observed=True).agg(
    customers              = ('user_id',               'nunique'),
    avg_orders_per_customer = ('unique_orders',         'mean'),
    median_orders_per_customer = ('unique_orders',     'median'),
    avg_unique_products    = ('unique_products',       'mean'),
    median_unique_products = ('unique_products',       'median'),
    avg_product_price      = ('avg_product_price',     'mean'),
    avg_reordered_items    = ('total_reordered_items', 'mean'),
    avg_reorder_rate       = ('reorder_rate',          'mean'),
    avg_income             = ('income',                'mean'),
    avg_age                = ('age',                   'mean'),
    avg_behaviour_score    = ('behaviour_score',       'mean'),
).reset_index()

file_03['customer_share'] = file_03['customers'] / file_03['customers'].sum()
file_03['priority_sort']  = file_03['priority_level'].map(priority_sort_map)
file_03 = file_03.sort_values('priority_sort').reset_index(drop=True)

assert file_03['customers'].sum() == 206209

save_csv(file_03, '03_priority_level_summary_FULL.csv')


Building file 3...
   ✅ 03_priority_level_summary_FULL.csv                              4 rows × 14 cols  (0.00 MB)


In [15]:
# ============================================================
# FILE 4 — BEHAVIOUR SEGMENT SUMMARY
# ============================================================

print()
print("Building file 4...")

file_04 = df_customers_agg.groupby('behaviour_segment', observed=True).agg(
    customers              = ('user_id',               'nunique'),
    avg_orders_per_customer = ('unique_orders',         'mean'),
    median_orders_per_customer = ('unique_orders',     'median'),
    avg_unique_products    = ('unique_products',       'mean'),
    median_unique_products = ('unique_products',       'median'),
    avg_product_price      = ('avg_product_price',     'mean'),
    avg_reordered_items    = ('total_reordered_items', 'mean'),
    avg_reorder_rate       = ('reorder_rate',          'mean'),
    avg_income             = ('income',                'mean'),
    avg_age                = ('age',                   'mean'),
    avg_behaviour_score    = ('behaviour_score',       'mean'),
).reset_index()

file_04['customer_share'] = file_04['customers'] / file_04['customers'].sum()

assert file_04['customers'].sum() == 206209

save_csv(file_04, '04_behaviour_segment_summary_FULL.csv')


Building file 4...
   ✅ 04_behaviour_segment_summary_FULL.csv                           9 rows × 13 cols  (0.00 MB)


In [16]:
# ============================================================
# FILE 5 — REGION × STRATEGIC SEGMENT SUMMARY
# ============================================================
# One row per (region, strategic_target_segment) combination.

print()
print("Building file 5...")

file_05 = df_customers_agg.groupby(
    ['region', 'strategic_target_segment'], observed=True
).agg(
    customers              = ('user_id',               'nunique'),
    avg_orders_per_customer = ('unique_orders',         'mean'),
    avg_unique_products    = ('unique_products',       'mean'),
    avg_product_price      = ('avg_product_price',     'mean'),
    avg_reordered_items    = ('total_reordered_items', 'mean'),
    avg_reorder_rate       = ('reorder_rate',          'mean'),
    avg_income             = ('income',                'mean'),
    avg_age                = ('age',                   'mean'),
    avg_behaviour_score    = ('behaviour_score',       'mean'),
).reset_index()

# Two share metrics for different dashboard cuts:
# - segment_region_share: within a region, what % does this segment contribute?
# - region_segment_share: within a segment, what % does this region contribute?
region_totals = file_05.groupby('region')['customers'].transform('sum')
segment_totals = file_05.groupby('strategic_target_segment', observed=True)['customers'].transform('sum')
file_05['segment_region_share'] = file_05['customers'] / region_totals
file_05['region_segment_share'] = file_05['customers'] / segment_totals

assert file_05['customers'].sum() == 206209

save_csv(file_05, '05_region_strategic_segment_summary_FULL.csv')


Building file 5...
   ✅ 05_region_strategic_segment_summary_FULL.csv                   44 rows × 13 cols  (0.01 MB)


/var/folders/kg/d6djty_168l6kzcc718bcldm0000gn/T/ipykernel_7810/4097158532.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  region_totals = file_05.groupby('region')['customers'].transform('sum')


In [17]:
# ============================================================
# FILE 6 — DEMOGRAPHIC × STRATEGIC SEGMENT SUMMARY
# ============================================================
# Most granular customer-level summary: every combination of
# strategic_target_segment × priority × age × income × household.

print()
print("Building file 6...")

file_06 = df_customers_agg.groupby([
    'strategic_target_segment', 'priority_level',
    'age_group', 'income_group', 'household_group'
], observed=True).agg(
    customers              = ('user_id',               'nunique'),
    avg_orders_per_customer = ('unique_orders',         'mean'),
    avg_unique_products    = ('unique_products',       'mean'),
    avg_product_price      = ('avg_product_price',     'mean'),
    avg_reordered_items    = ('total_reordered_items', 'mean'),
    avg_reorder_rate       = ('reorder_rate',          'mean'),
    avg_income             = ('income',                'mean'),
    avg_age                = ('age',                   'mean'),
    avg_behaviour_score    = ('behaviour_score',       'mean'),
).reset_index()

file_06['customer_share'] = file_06['customers'] / file_06['customers'].sum()

assert file_06['customers'].sum() == 206209

save_csv(file_06, '06_demographic_strategic_segment_summary_FULL.csv')


Building file 6...
   ✅ 06_demographic_strategic_segment_summary_FULL.csv             212 rows × 15 cols  (0.05 MB)


In [18]:
# ============================================================
# FILE 7 — DEPARTMENT TARGETING (transaction-aggregated)
# ============================================================
# Summary of transaction activity by segment and department.
# Built from df_master_segmented, the 32M-row transaction file.

print()
print("Building file 7 (this takes a moment — 32M rows)...")

file_07 = df_master_segmented.groupby([
    'strategic_target_segment', 'priority_level',
    'behaviour_segment', 'department_name'
], observed=True).agg(
    rows             = ('order_id',   'size'),
    unique_customers = ('user_id',    'nunique'),
    unique_orders    = ('order_id',   'nunique'),
    avg_price        = ('prices',     'mean'),
    reorder_rate     = ('reordered',  'mean')
).reset_index().rename(columns={'department_name': 'department'})

# Total rows must match transaction count
assert file_07['rows'].sum() == df_master_segmented.shape[0]

save_csv(file_07, '07_department_targeting_summary_FULL.csv')


Building file 7 (this takes a moment — 32M rows)...
   ✅ 07_department_targeting_summary_FULL.csv                    1,473 rows ×  9 cols  (0.18 MB)


In [19]:
# ============================================================
# FILE 8 — PRODUCT TARGETING (transaction-aggregated)
# ============================================================
# Most granular: segment × department × product.
# This file will be large (~hundreds of thousands of rows).

print()
print("Building file 8 (largest file — also takes a moment)...")

file_08 = df_master_segmented.groupby([
    'strategic_target_segment', 'priority_level',
    'behaviour_segment', 'department_name',
    'product_id', 'product_name'
], observed=True).agg(
    rows             = ('order_id',   'size'),
    unique_customers = ('user_id',    'nunique'),
    unique_orders    = ('order_id',   'nunique'),
    avg_price        = ('prices',     'mean'),
    reorder_rate     = ('reordered',  'mean')
).reset_index().rename(columns={'department_name': 'department'})

assert file_08['rows'].sum() == df_master_segmented.shape[0]

save_csv(file_08, '08_product_targeting_summary_FULL.csv')


Building file 8 (largest file — also takes a moment)...
   ✅ 08_product_targeting_summary_FULL.csv                   1,116,889 rows × 11 cols  (148.09 MB)


In [20]:
# ============================================================
# DONE
# ============================================================

print()
print("✅ NOTEBOOK 9 COMPLETE — all 8 files exported")
print(f"   Output folder: {output_dir}")


✅ NOTEBOOK 9 COMPLETE — all 8 files exported
   Output folder: /Users/elia/Desktop/[02] Data Projects/[2] Working Folder/[3] InstaCart Store/[2] Data/[3] Power BI Exports
